# HAR Real-Data Experiment

This notebook runs the HAR real-data experiment and reports a covariate balancedness diagnostic for the subject-level tasks.

Protocol summary:
- raw 561-dimensional HAR features with global Min-Max scaling;
- subjects are treated as tasks;
- binary label: `STANDING` versus all other activities;
- 30 random train/test splits;
- 5-fold cross-validation for `ARMUL` and `OURS`.


In [1]:
from pathlib import Path
import sys


def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except Exception:
        return False
    return True


if _running_in_colab():
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

_code_candidates = [
    Path.cwd(),
    Path.cwd() / "MTLR_Codes",
    Path("/content/drive/MyDrive/Colab Notebooks/MTLR_Codes"),
    Path("/content/drive/My Drive/Colab Notebooks/MTLR_Codes"),
    Path("/content/drive/MyDrive/MTLR_Codes"),
    Path("/content/drive/My Drive/MTLR_Codes"),
]

CODE_DIR = None
for candidate in _code_candidates:
    if (candidate / "path_setup.py").exists():
        CODE_DIR = candidate.resolve()
        break

if CODE_DIR is None:
    raise FileNotFoundError(
        "Could not locate MTLR_Codes. In Colab, upload the folder to "
        "MyDrive/Colab Notebooks/MTLR_Codes."
    )

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from path_setup import setup_project_paths

CODE_DIR, PROJECT_ROOT, FIGURE_DIR = setup_project_paths(chdir=True)
print(f"CODE_DIR    : {CODE_DIR}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FIGURE_DIR  : {FIGURE_DIR}")

import importlib
import pandas as pd

pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 20)

import real_data_har as har
importlib.reload(har)

print(f"Positive HAR label(s): {har.POSITIVE_LABELS}")
print(f"Default q-grid: {har.Q_GRID}")


Mounted at /content/drive
CODE_DIR    : /content/drive/MyDrive/Colab Notebooks/MTLR_Codes
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks
FIGURE_DIR  : /content/drive/MyDrive/Colab Notebooks/MTLR_Codes/Images
Positive HAR label(s): [5]
Default q-grid: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]


In [2]:
DATA_DIR = CODE_DIR
Q_GRID = list(har.Q_GRID)
N_SPLITS = har.N_SPLITS
N_FOLD = har.N_FOLD
ETA = har.ETA
MAXITER = har.MAXITER

# For a quick Colab smoke test, temporarily set `N_SPLITS = 1`.
har.Q_GRID = Q_GRID
har.N_SPLITS = N_SPLITS
har.N_FOLD = N_FOLD
har.ETA = ETA
har.MAXITER = MAXITER

print("Configuration")
print("-------------")
print(f"DATA_DIR : {DATA_DIR}")
print(f"Q_GRID   : {Q_GRID}")
print(f"N_SPLITS : {N_SPLITS}")
print(f"N_FOLD   : {N_FOLD}")
print(f"ETA      : {ETA}")
print(f"MAXITER  : {MAXITER}")


Configuration
-------------
DATA_DIR : /content/drive/MyDrive/Colab Notebooks/MTLR_Codes
Q_GRID   : [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
N_SPLITS : 30
N_FOLD   : 5
ETA      : 0.1
MAXITER  : 200


## HAR Covariate Balancedness Diagnostic

This diagnostic summarizes how uneven the subject-specific empirical second moments are across HAR subjects. For each subject/task, let
$$
\Sigma_j = \frac{1}{n_j} X_j^\top X_j.
$$
For the HAR data, we estimate the aggregate covariance in the balancedness definition by the empirical second moment over all HAR covariate samples,
$$
\Sigma_{\mathrm{all}} = \frac{1}{N}\sum_{j=1}^m X_j^\top X_j.
$$
The notebook reports the single plug-in balancedness estimate
$$
B_{\mathrm{HAR}} = \max_j \lambda_{\max}\!\left(\Sigma_{\mathrm{all}}^{\dagger/2}\Sigma_j\Sigma_{\mathrm{all}}^{\dagger/2}\right),
$$
with the generalized eigenvalue computed on the empirical range of \(\Sigma_{\mathrm{all}}\). Labels are not used here; balancedness only depends on the covariates.


In [3]:
import numpy as np
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
from path_setup import find_har_dataset


def load_har_subject_covariates(dataset_root):
    X_train = np.loadtxt(dataset_root / "train" / "X_train.txt")
    sub_train = np.loadtxt(dataset_root / "train" / "subject_train.txt").astype(int)
    X_test = np.loadtxt(dataset_root / "test" / "X_test.txt")
    sub_test = np.loadtxt(dataset_root / "test" / "subject_test.txt").astype(int)

    X_all = np.concatenate([X_train, X_test], axis=0)
    sub_all = np.concatenate([sub_train, sub_test], axis=0)

    # Use the same global feature scaling as the HAR experiment.
    X_scaled = MinMaxScaler(feature_range=(0, 1)).fit_transform(X_all)

    subject_ids = sorted(np.unique(sub_all).tolist())
    X_list = [X_scaled[sub_all == subject_id] for subject_id in subject_ids]
    return subject_ids, X_list


def empirical_second_moments(X_list):
    return [(X.T @ X) / X.shape[0] for X in X_list]


def max_generalized_eigenvalue(A, B, tol=1e-10):
    vals, vecs = np.linalg.eigh(B)
    keep = vals > tol
    if not np.any(keep):
        return np.inf
    B_inv_sqrt = (vecs[:, keep] / np.sqrt(vals[keep])) @ vecs[:, keep].T
    return float(np.linalg.eigvalsh(B_inv_sqrt @ A @ B_inv_sqrt).max())


def har_balancedness_diagnostic(code_dir):
    dataset_root = find_har_dataset(code_dir)
    subject_ids, X_list = load_har_subject_covariates(dataset_root)
    sigmas = empirical_second_moments(X_list)

    # Plug-in estimate of Sigma_S: use the empirical second moment over all HAR samples.
    total_n = sum(X.shape[0] for X in X_list)
    sigma_all = sum(X.shape[0] * S for X, S in zip(X_list, sigmas)) / total_n

    rows = []
    for subject_id, X, S in zip(subject_ids, X_list, sigmas):
        rows.append(
            {
                "subject": subject_id,
                "n": X.shape[0],
                "B_to_Sigma_all": max_generalized_eigenvalue(S, sigma_all),
                "trace_sigma": float(np.trace(S)),
            }
        )

    by_subject = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "m": len(subject_ids),
                "min_n": int(min(X.shape[0] for X in X_list)),
                "max_n": int(max(X.shape[0] for X in X_list)),
                "B_HAR": float(by_subject["B_to_Sigma_all"].max()),
            }
        ]
    )
    return dataset_root, summary, by_subject


dataset_root, b_summary, b_by_subject = har_balancedness_diagnostic(DATA_DIR)
print(f"Dataset root: {dataset_root}")
print("Summary:")
display(b_summary.round(4))
print("Largest subject-level values:")
display(b_by_subject.sort_values("B_to_Sigma_all", ascending=False).head(10).round(4))


Dataset root: /content/drive/MyDrive/Colab Notebooks/MTLR_Codes/UCI_HAR_Dataset
Summary:


,m,min_n,max_n,B_HAR
0,30,281,409,28.8287


Largest subject-level values:


,subject,n,B_to_Sigma_all,trace_sigma
13,14,323,28.8287,79.8982
9,10,294,28.7067,76.2987
18,19,360,26.2524,80.6579
7,8,281,26.1228,80.6453
5,6,325,25.6657,79.5967
22,23,372,25.6554,79.5811
24,25,409,23.8227,66.8226
6,7,308,23.6316,75.3306
21,22,321,22.1286,75.2584
19,20,354,22.0842,76.5863


In [ ]:
results_df = har.run_experiment(DATA_DIR)

summary = results_df[["DP", "ITL", "ARMUL", "OURS"]].agg(["mean", "std"]).T
summary.columns = ["Mean Error", "Std Dev"]
summary


Loaded HAR data with m=30 tasks and d=561 raw features.
Starting experiment: 30 random splits with 5-fold CV.
--- Split 1/30 ---
   [Split 0] ARMUL: 0.0630 | OURS: 0.0132
   [Split 1] ARMUL: 0.0484 | OURS: 0.0106
   [Split 2] ARMUL: 0.0440 | OURS: 0.0121
   [Split 3] ARMUL: 0.0498 | OURS: 0.0109
   [Split 4] ARMUL: 0.0513 | OURS: 0.0067
--- Split 6/30 ---
   [Split 5] ARMUL: 0.0532 | OURS: 0.0145
   [Split 6] ARMUL: 0.0513 | OURS: 0.0126
   [Split 7] ARMUL: 0.0537 | OURS: 0.0093
   [Split 8] ARMUL: 0.0503 | OURS: 0.0101
   [Split 9] ARMUL: 0.0523 | OURS: 0.0141
--- Split 11/30 ---
   [Split 10] ARMUL: 0.0523 | OURS: 0.0117
   [Split 11] ARMUL: 0.0562 | OURS: 0.0141
   [Split 12] ARMUL: 0.0518 | OURS: 0.0139
   [Split 13] ARMUL: 0.0552 | OURS: 0.0144
   [Split 14] ARMUL: 0.0581 | OURS: 0.0140
--- Split 16/30 ---
   [Split 15] ARMUL: 0.0498 | OURS: 0.0162
   [Split 16] ARMUL: 0.0498 | OURS: 0.0115
   [Split 17] ARMUL: 0.0449 | OURS: 0.0081
   [Split 18] ARMUL: 0.0572 | OURS: 0.0129
   [S

,Mean Error,Std Dev
DP,0.0761,0.0046
ITL,0.0467,0.0051
ARMUL,0.0524,0.0043
OURS,0.0125,0.0032
